# **Downloading Libraries**

In [1]:
!pip install --upgrade streamlit pyngrok transformers accelerate --quiet --no-deps --ignore-installed blinker


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.1/367.1 kB 19.3 MB/s eta 0:00:00


In [2]:
!pip install -U transformers huggingface-hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.8/558.8 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.33.5
    Uninstalling huggingface-hub-0.33.5:
      Successfully uninstalled huggingface-hub-0.33.5


**Importing libraries and writing main code**



In [14]:
# Streamlit chatbot for general health questions using TinyLlama

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import textwrap # Import textwrap for formatting

# Load model and tokenizer only once
# @st.cache_resource # Removed Streamlit cache decorator
def load_model():
    model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    return pipeline("text-generation", model=model, tokenizer=tokenizer)

# Build prompt with clear instructions
def build_prompt(user_input):
    return f"""<s>[INST] You are a friendly medical assistant. Provide general, safe health information in simple language. Avoid giving any harmful or risky advice, and always remind the user to consult a licensed doctor for serious concerns.

User: {user_input}
Assistant:"""

# Filter out dangerous queries
def is_safe_query(query):
    harmful_keywords = [
        "suicide", "overdose", "kill", "die", "death", "emergency",
        "diagnose", "prescribe", "surgery", "bleeding", "poison", "inject"
    ]
    return not any(word in query.lower() for word in harmful_keywords)

# Get model response
def get_response(pipe, user_input):
    if not is_safe_query(user_input):
        return "⚠️ I'm not able to help with that. Please contact a licensed healthcare professional."
    prompt = build_prompt(user_input)
    result = pipe(prompt, max_new_tokens=512, temperature=0.7, do_sample=True)[0]["generated_text"] # Increased max_new_tokens

    # Remove unwanted token echoes
    if "Assistant:" in result:
        response = result.split("Assistant:")[-1].strip()
        response = response.split("</s>")[0].strip()
        return response
    return result.strip()

# Load the model once
pipe = load_model()



Device set to use cpu


In [16]:
while True:
    user_input = input("You: ")
    if not user_input:
        break  # Exit the loop if the user enters an empty line
    print("Assistant: ", get_response(pipe, user_input))

You: i am having fever and cold give me tips
Assistant:  yes, here are some general tips to combat the symptoms of a fever:

1. Drink plenty of fluids: your body needs water to flush out the fluids that rise due to fever. Try to drink as much water as possible.

2. Eat hot food: warming foods like soup or hot chocolate can help numb the body's temperature.

3. Drink lemon water: lemon water can help lower the body's core temperature by diluting the blood and promoting sweating.

4. Take medication: if you feel feverish, you can take fever-reducing medications like acetaminophen, ibuprofen, or paracetamol.

5. Avoid hot surfaces: avoid touching or sitting on hot surfaces, such as stoves or heaters.

6. Don't shower near a toilet: don't shower near a toilet seat, as it can cause a panic attack. Use a separate showerhead away from the toilet seat.

7. Get plenty of sleep: sleep plays a vital role in fighting fever. Aim for at least 7-8 hours of sleep every night.

8. Stay hydrated: drink 